# Benchmark: parallel polarization DWTs and one-arm link blocking

This notebook evaluates two numerically equivalent runtime changes for the
full-year prepared one-arm response:

1. sequential versus two-thread plus/cross WDM transforms;
2. the production sequential six-link shift versus an experimental
   analytic-parity kernel that processes small link blocks together.

The production dispatcher is not changed by the experiment.  Link-block
candidates are called through a separately named plan method.

The primary candidate set keeps approximately

\[
N_{\rm link\,block}\,N_{\rm row\,chunk}=32640,
\]

so increasing link parallelism does not simultaneously increase the dominant
link-row-lag-frequency working set.  More aggressive full-row candidates are
available but disabled by default.


In [ ]:
# --- 1. Imports, environment, and source-file checks ---

import gc
import inspect
import os
import platform
import sys
import time
from datetime import datetime
from itertools import product

# ------------------------------------------------------------------
# Backend selection
#
# IMPORTANT:
# This must be set before importing jax, jax.numpy, FEW, WDM,
# WDM_LISAresponse, or anything else that may import JAX.
# ------------------------------------------------------------------

os.environ["JAX_PLATFORMS"] = "cpu"

import jax
import jax.numpy as jnp
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import psutil
except ImportError:
    psutil = None

from few.utils.constants import YRSID_SI
from few.waveform import GenerateEMRIWaveform
from lisaorbits.file import read_orbit_file
from lisaorbits import InterpolatedOrbits

from WDM_LISAresponse.code.lisa_response.waveforms import (
    generate_windowed_padded_waveform,
)
from WDM_LISAresponse.code.lisa_response.response_with_plan import (
    _build_wdm_dT_grid,
    _build_wdm_dT_context_from_wdm,
    _transform_wdm_polarizations,
)
from WDM_LISAresponse.code.lisa_response.one_arm_plan import (
    OneArmWDMPlan,
    _normalise_one_arm_execution_mode,
)

import WDM_LISAresponse.code.lisa_response.one_arm_plan as one_arm_plan_module
import WDM_LISAresponse.code.lisa_response.response_with_plan as response_with_plan_module
import WDM_LISAresponse
import WDM


_process = psutil.Process(os.getpid()) if psutil is not None else None


def rss_gib():
    if _process is None:
        return np.nan
    return _process.memory_info().rss / 1024**3


print("Timestamp:", datetime.now().isoformat(timespec="seconds"))
print("Python executable:", sys.executable)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("JAX:", jax.__version__)
print("Requested JAX platforms:", os.environ.get("JAX_PLATFORMS"))
print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
print("WDM:", WDM.__file__)
print("WDM_LISAresponse:", WDM_LISAresponse.__file__)
print("one_arm_plan:", one_arm_plan_module.__file__)
print("response_with_plan:", response_with_plan_module.__file__)
print(
    "OneArmWDMPlan.apply_device:",
    inspect.signature(OneArmWDMPlan.apply_device),
)
print("RSS:", f"{rss_gib():.2f} GiB")

assert jax.default_backend() == "cpu", (
    "JAX did not initialize the CPU backend. Restart the kernel and "
    "run this cell first, before any JAX-related imports."
)

assert all(device.platform == "cpu" for device in jax.devices())

assert (
    _normalise_one_arm_execution_mode("materialized")
    == "materialized"
)
assert (
    "execution_mode"
    in inspect.signature(OneArmWDMPlan.apply_device).parameters
)

from WDM.code.time_delay_filters import _time_shift_jax as time_shift_jax_module

assert "parallel_wdm_transforms" in inspect.signature(
    _build_wdm_dT_context_from_wdm
).parameters
assert hasattr(
    OneArmWDMPlan,
    "apply_batch_device_materialized_linkblock",
)
assert hasattr(
    time_shift_jax_module,
    "assemble_shift_target_batch_chunked_lagblock_linkblock_jax",
)


In [ ]:
# --- 2. Scientific configuration and benchmark controls ---

# Full-year production-like case.
N = 3_200_000
dt = 10
Nf = 100
q_wdm = 4
d_wdm = 4
A_frac = 0.25
L_TRUNC = 25

# Windowing.
alpha = 0.005
tukey_factor = 4

# Orbit start and sky position.
ORBIT_FILE = "esa-trailing-orbits.h5"
theta = np.pi / 2
phi = 0.0

# EMRI parameters.
m1 = 1e6
m2 = 10.0
a = 0.9
p0 = 8.5
e0 = 0.3
x0 = 1.0
dist = 1.0
qK = np.pi / 3
phiK = np.pi / 3
Phi_phi0 = np.pi / 3
Phi_theta0 = 0.0
Phi_r0 = np.pi / 3
eps = 1e-2

BASE_SHIFT_KWARGS = {
    "assembly_backend": "lagfirst_chunked_lagblock",
    "assembly_precision": "complex64",
    "tl_tp_mode": "interp",
    "tl_tp_interp_points": 32,
    "tl_tp_interp_kind": "linear",
    "use_jax": True,
    "row_chunk_size": 32640,
    "lag_block_size": 17,
    "job_block_size": 1,
}

BASELINE_ROW_CHUNK = 32640
BASELINE_LAG_BLOCK = 17

DWT_WARMUPS = 2
DWT_REPEATS = 15
SHIFT_WARMUPS = 2
SHIFT_REPEATS = 15
PIPELINE_WARMUPS = 2
PIPELINE_REPEATS = 11

MIN_MEDIAN_IMPROVEMENT_PERCENT = 3.0
MAX_REL_L2 = 1e-6

# Keep link_block * row_chunk approximately equal to the production row chunk.
SAFE_LINKBLOCK_CANDIDATES = [
    (1, 32640),
    (2, 16320),
    (3, 10880),
    (6, 5440),
]

# These increase the logical active working set and may be much slower or OOM.
RUN_AGGRESSIVE_LINKBLOCKS = False
AGGRESSIVE_LINKBLOCK_CANDIDATES = [
    (2, 32640),
    (3, 32640),
    (6, 32640),
]

RUN_COMPILED_MEMORY_ANALYSIS = True

print("Expected WDM Nt:", N // (2 * Nf))
print("Lags:", 2 * L_TRUNC + 1)
print("Safe candidates:", SAFE_LINKBLOCK_CANDIDATES)
print("Aggressive candidates enabled:", RUN_AGGRESSIVE_LINKBLOCKS)


In [ ]:
# --- 3. Timing, validation, and display helpers ---

def block_result(value):
    # Synchronize all JAX leaves before stopping a timer.
    for leaf in jax.tree_util.tree_leaves(value):
        if hasattr(leaf, 'block_until_ready'):
            leaf.block_until_ready()
    return value


def summarize_timings(timings):
    timings = np.asarray(timings, dtype=float)
    if timings.size == 0:
        raise ValueError('At least one timing is required.')

    q25_s, q75_s = np.percentile(timings, [25.0, 75.0])

    return {
        'median_s': float(np.median(timings)),
        'minimum_s': float(np.min(timings)),
        'maximum_s': float(np.max(timings)),
        'mean_s': float(np.mean(timings)),
        'std_s': float(np.std(timings)),
        'q25_s': float(q25_s),
        'q75_s': float(q75_s),
        'iqr_s': float(q75_s - q25_s),
        'times_s': [float(x) for x in timings],
    }


def benchmark_call(fn, *, warmups=1, repeats=3):
    # Warm, synchronize, and return all individual runtimes.
    warmups = max(0, int(warmups))
    repeats = max(1, int(repeats))

    result = None
    for _ in range(warmups):
        result = fn()
        block_result(result)

    timings = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = fn()
        block_result(result)
        timings.append(time.perf_counter() - t0)

    return result, summarize_timings(timings)


def compare_device_batches(candidate, reference):
    # Compare two device-resident six-link batches using scalar reductions.
    candidate = jnp.asarray(candidate)
    reference = jnp.asarray(reference)

    delta = candidate - reference
    max_abs_value = jnp.max(jnp.abs(delta))
    ref_norm = jnp.linalg.norm(reference)
    rel_l2_value = jnp.where(
        ref_norm > 0,
        jnp.linalg.norm(delta) / ref_norm,
        jnp.nan,
    )

    max_abs_value.block_until_ready()
    rel_l2_value.block_until_ready()

    return {
        'max_abs': float(max_abs_value),
        'rel_l2': float(rel_l2_value),
    }


def assess_paired_candidate(
    baseline_times,
    candidate_times,
    *,
    rel_l2,
    min_improvement_percent=MIN_MEDIAN_IMPROVEMENT_PERCENT,
    max_rel_l2=MAX_REL_L2_FOR_SCHEDULING,
):
    baseline = summarize_timings(baseline_times)
    candidate = summarize_timings(candidate_times)

    runtime_change_percent = (
        100.0 * (candidate['median_s'] / baseline['median_s'] - 1.0)
    )
    improvement_percent = -runtime_change_percent
    numerically_valid = bool(np.isfinite(rel_l2) and rel_l2 <= max_rel_l2)

    # Conservative distributional guard:
    # at least 75% of candidate timings must be below at least 75% of baseline
    # timings. Equivalently, the central 50% intervals do not overlap.
    iqr_separated = bool(candidate['q75_s'] < baseline['q25_s'])

    if (
        numerically_valid
        and improvement_percent >= min_improvement_percent
        and iqr_separated
    ):
        decision = 'adopt'
    elif numerically_valid and improvement_percent >= min_improvement_percent:
        decision = 'retest'
    else:
        decision = 'retain_baseline'

    return {
        'baseline': baseline,
        'candidate': candidate,
        'runtime_change_percent': float(runtime_change_percent),
        'improvement_percent': float(improvement_percent),
        'numerically_valid': numerically_valid,
        'iqr_separated': iqr_separated,
        'decision': decision,
    }


def make_shift_kwargs(
    row_chunk_size,
    lag_block_size,
    *,
    backend='lagfirst_chunked_lagblock',
    job_block_size=1,
):
    kwargs = dict(BASE_SHIFT_KWARGS)
    kwargs.update({
        'assembly_backend': str(backend),
        'row_chunk_size': int(row_chunk_size),
        'lag_block_size': int(lag_block_size),
        'job_block_size': int(job_block_size),
    })
    return kwargs


def build_one_arm_plan(grid_context, kwargs):
    t0 = time.perf_counter()
    plan = OneArmWDMPlan.build_from_context(
        grid_context,
        orbits_trailing,
        theta,
        phi,
        L_trunc=L_TRUNC,
        variable_shift_kwargs=kwargs,
        use_jax=True,
    )
    build_s = time.perf_counter() - t0
    return plan, float(build_s)


def show_rows(rows, *, sort_by=None):
    if pd is not None:
        frame = pd.DataFrame(rows)
        if sort_by is not None and sort_by in frame.columns:
            frame = frame.sort_values(sort_by, kind='stable').reset_index(drop=True)
        display(frame)
        return frame

    rows_out = list(rows)
    if sort_by is not None:
        rows_out = sorted(rows_out, key=lambda row: row.get(sort_by, np.inf))
    for row in rows_out:
        print(row)
    return rows_out


def save_rows(rows, filename):
    if not SAVE_RESULTS:
        return
    if pd is None:
        print('pandas is unavailable; skipping CSV export:', filename)
        return
    pd.DataFrame(rows).to_csv(filename, index=False)
    print('Saved:', filename)


## 4. Generate the fixed waveform and orbit object

In [ ]:
# --- 4a. Orbit and waveform generator ---

td_gen = GenerateEMRIWaveform(
    'FastKerrEccentricEquatorialFlux',
    sum_kwargs=dict(pad_output=True, odd_len=True),
    return_list=True,
)

orbit_data = read_orbit_file(ORBIT_FILE)
orbit_times = orbit_data.t0 + orbit_data.dt * np.arange(orbit_data.size)
orbits_trailing = InterpolatedOrbits(
    orbit_times,
    orbit_data.positions,
    orbit_data.velocities,
)

t0 = orbit_data.t0 + 1.0 * YRSID_SI

model_kwargs = {
    'td_gen': td_gen,
    'm1': m1,
    'm2': m2,
    'a': a,
    'p0': p0,
    'e0': e0,
    'x0': x0,
    'dist': dist,
    'theta': theta,
    'phi': phi,
    'qK': qK,
    'phiK': phiK,
    'Phi_phi0': Phi_phi0,
    'Phi_theta0': Phi_theta0,
    'Phi_r0': Phi_r0,
    'eps': eps,
}

window_kwargs = {
    'alpha': alpha,
    'tukey_factor': tukey_factor,
}

print('Orbit samples:', orbit_data.size)
print('Starting waveform generation...')


In [ ]:
# --- 4b. Generate and pad the waveform ---

t_wave0 = time.perf_counter()

times_pad, hplus_td, hcross_td, pad_info = generate_windowed_padded_waveform(
    waveform_model='emri',
    t0=t0,
    dt=dt,
    N=N,
    model_kwargs=model_kwargs,
    window_function='tukey',
    window_kwargs=window_kwargs,
    wdm_Nf=Nf,
    return_padding_info=True,
)

waveform_generation_s = time.perf_counter() - t_wave0

print('Waveform generation_s:', f'{waveform_generation_s:.3f}')
print('times_pad:', times_pad.shape, times_pad.dtype)
print('hplus_td:', hplus_td.shape, hplus_td.dtype)
print('hcross_td:', hcross_td.shape, hcross_td.dtype)
print('WDM Nt x Nf:', pad_info['wdm_Nt'], 'x', Nf)
print('Tail right padding:', pad_info['wdm_tail_pad_right'])
print('RSS:', f'{rss_gib():.2f} GiB')


## 5. Build one WDM grid and compare sequential/parallel DWTs

The transform object and padded waveforms are identical.  Only the scheduling
of the independent plus/cross calls changes.


In [ ]:
# --- 5. DWT comparison ---

t_grid0 = time.perf_counter()
grid_context = _build_wdm_dT_grid(
    times_pad,
    Nf=Nf,
    d=d_wdm,
    A_frac=A_frac,
    q=q_wdm,
    calc_m0=True,
)
grid_build_s = time.perf_counter() - t_grid0
wdm = grid_context["wdm"]


def run_sequential_context():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        wdm,
        parallel_wdm_transforms=False,
    )


def run_parallel_context():
    return _build_wdm_dT_context_from_wdm(
        times_pad,
        hplus_td,
        hcross_td,
        wdm,
        parallel_wdm_transforms=True,
    )

# Compile/warm both routes before paired timing.
for _ in range(DWT_WARMUPS):
    sequential_context = run_sequential_context()
    parallel_context = run_parallel_context()
    block_result((
        sequential_context["w_t_plus"],
        sequential_context["w_t_cross"],
        parallel_context["w_t_plus"],
        parallel_context["w_t_cross"],
    ))

sequential_context = run_sequential_context()
parallel_context = run_parallel_context()
block_result((
    sequential_context["w_t_plus"],
    sequential_context["w_t_cross"],
    parallel_context["w_t_plus"],
    parallel_context["w_t_cross"],
))

plus_comparison = compare_device_batches(
    jnp.asarray(parallel_context["w_t_plus"])[None, ...],
    jnp.asarray(sequential_context["w_t_plus"])[None, ...],
)
cross_comparison = compare_device_batches(
    jnp.asarray(parallel_context["w_t_cross"])[None, ...],
    jnp.asarray(sequential_context["w_t_cross"])[None, ...],
)

sequential_times = []
parallel_times = []

for repeat_index in range(DWT_REPEATS):
    order = (
        (("sequential", run_sequential_context), ("parallel", run_parallel_context))
        if repeat_index % 2 == 0
        else (("parallel", run_parallel_context), ("sequential", run_sequential_context))
    )

    for label, function in order:
        started = time.perf_counter()
        context = function()
        block_result((context["w_t_plus"], context["w_t_cross"]))
        elapsed = time.perf_counter() - started
        if label == "sequential":
            sequential_times.append(float(elapsed))
        else:
            parallel_times.append(float(elapsed))

sequential_dwt = summarize_timings(sequential_times)
parallel_dwt = summarize_timings(parallel_times)
dwt_assessment = assess_paired_candidate(
    sequential_times,
    parallel_times,
    rel_l2=max(plus_comparison["rel_l2"], cross_comparison["rel_l2"]),
)

print("WDM grid build_s:", f"{grid_build_s:.3f}")
print("Sequential DWT median_s:", f"{sequential_dwt['median_s']:.3f}")
print("Parallel DWT median_s:", f"{parallel_dwt['median_s']:.3f}")
print("Runtime change:", f"{dwt_assessment['runtime_change_percent']:+.2f}%")
print("IQR separated:", dwt_assessment["iqr_separated"])
print("Plus comparison:", plus_comparison)
print("Cross comparison:", cross_comparison)
print("Decision:", dwt_assessment["decision"])
print("Sequential all:", [round(x, 3) for x in sequential_times])
print("Parallel all:", [round(x, 3) for x in parallel_times])

wave_context = sequential_context
w_plus_device = jnp.asarray(wave_context["w_t_plus"], dtype=jnp.complex64)
w_cross_device = jnp.asarray(wave_context["w_t_cross"], dtype=jnp.complex64)
block_result((w_plus_device, w_cross_device))


## 6. Establish the production one-arm baseline

In [ ]:
# --- 6. Production baseline ---

baseline_plan, baseline_plan_build_s = build_one_arm_plan(
    grid_context,
    BASE_SHIFT_KWARGS,
)


def run_baseline_shift():
    return baseline_plan.apply_batch_device_materialized(
        w_plus_device,
        w_cross_device,
    )

reference_device, baseline_shift_timing = benchmark_call(
    run_baseline_shift,
    warmups=SHIFT_WARMUPS,
    repeats=SHIFT_REPEATS,
)

print("Plan build_s:", f"{baseline_plan_build_s:.3f}")
print("Baseline shift median_s:", f"{baseline_shift_timing['median_s']:.3f}")
print("Baseline IQR_s:", f"{baseline_shift_timing['iqr_s']:.3f}")
print("Reference:", reference_device.shape, reference_device.dtype)
print(
    "Persistent plan memory_MiB:",
    baseline_plan.shift_plan.device_plan_memory_bytes / 1024**2,
)


## 7. Screen analytic link-block candidates

The safe candidates reduce the row chunk as the link block grows.  This keeps
`link_block_size * row_chunk_size` fixed and tests whether extra link-level
parallelism helps without deliberately enlarging the dominant rank-four
working set.


In [ ]:
# --- 7. Link-block paired benchmark ---

candidates = list(SAFE_LINKBLOCK_CANDIDATES)
if RUN_AGGRESSIVE_LINKBLOCKS:
    candidates.extend(AGGRESSIVE_LINKBLOCK_CANDIDATES)

linkblock_rows = []
linkblock_raw = {}

for link_block_size, row_chunk_size in candidates:
    print(
        "\nCandidate:",
        f"link_block={link_block_size}",
        f"row_chunk={row_chunk_size}",
    )

    def run_candidate(
        link_block_size=link_block_size,
        row_chunk_size=row_chunk_size,
    ):
        return baseline_plan.apply_batch_device_materialized_linkblock(
            w_plus_device,
            w_cross_device,
            link_block_size=link_block_size,
            row_chunk_size=row_chunk_size,
            lag_block_size=BASELINE_LAG_BLOCK,
        )

    for _ in range(SHIFT_WARMUPS):
        block_result(run_baseline_shift())
        block_result(run_candidate())

    candidate_output = run_candidate()
    block_result(candidate_output)
    comparison = compare_device_batches(candidate_output, reference_device)

    baseline_times = []
    candidate_times = []

    for repeat_index in range(SHIFT_REPEATS):
        order = (
            (("baseline", run_baseline_shift), ("candidate", run_candidate))
            if repeat_index % 2 == 0
            else (("candidate", run_candidate), ("baseline", run_baseline_shift))
        )

        for label, function in order:
            started = time.perf_counter()
            output = function()
            block_result(output)
            elapsed = time.perf_counter() - started
            if label == "baseline":
                baseline_times.append(float(elapsed))
            else:
                candidate_times.append(float(elapsed))

    assessment = assess_paired_candidate(
        baseline_times,
        candidate_times,
        rel_l2=comparison["rel_l2"],
    )
    baseline_stats = assessment["baseline"]
    candidate_stats = assessment["candidate"]

    row = {
        "link_block_size": int(link_block_size),
        "row_chunk_size": int(row_chunk_size),
        "lag_block_size": int(BASELINE_LAG_BLOCK),
        "active_link_rows": int(link_block_size * row_chunk_size),
        "baseline_median_s": baseline_stats["median_s"],
        "candidate_median_s": candidate_stats["median_s"],
        "runtime_change_percent": assessment["runtime_change_percent"],
        "speedup": baseline_stats["median_s"] / candidate_stats["median_s"],
        "baseline_q25_s": baseline_stats["q25_s"],
        "baseline_q75_s": baseline_stats["q75_s"],
        "candidate_q25_s": candidate_stats["q25_s"],
        "candidate_q75_s": candidate_stats["q75_s"],
        "candidate_iqr_s": candidate_stats["iqr_s"],
        "iqr_separated": assessment["iqr_separated"],
        "max_abs": comparison["max_abs"],
        "rel_l2": comparison["rel_l2"],
        "decision": assessment["decision"],
    }
    linkblock_rows.append(row)
    linkblock_raw[(link_block_size, row_chunk_size)] = {
        "baseline_times_s": baseline_times,
        "candidate_times_s": candidate_times,
    }

    print("  baseline median:", f"{baseline_stats['median_s']:.3f} s")
    print("  candidate median:", f"{candidate_stats['median_s']:.3f} s")
    print("  runtime change:", f"{assessment['runtime_change_percent']:+.2f}%")
    print("  IQR separated:", assessment["iqr_separated"])
    print("  rel_l2:", f"{comparison['rel_l2']:.3e}")
    print("  decision:", assessment["decision"])

linkblock_table = show_rows(linkblock_rows, sort_by="candidate_median_s")


## 8. Optional compiled-memory analysis

Where supported by the installed JAX/XLA version, this section lowers the
production analytic-parity batch core and the experimental link-block core and
prints their executable memory analyses.  These figures describe compiled
buffers, not operating-system RSS.


In [ ]:
# --- 8. Compiled executable memory analysis ---

memory_rows = []


def memory_analysis_to_dict(value):
    if value is None:
        return {}
    result = {}
    for name in dir(value):
        if name.startswith("_"):
            continue
        try:
            item = getattr(value, name)
        except Exception:
            continue
        if callable(item):
            continue
        if isinstance(item, (int, float, np.integer, np.floating)):
            result[name] = int(item) if isinstance(item, (int, np.integer)) else float(item)
    if not result:
        result["repr"] = repr(value)
    return result


if RUN_COMPILED_MEMORY_ANALYSIS:
    try:
        amplitude_plus, amplitude_cross = baseline_plan._device_materialized_amplitudes()
        unshifted_device = one_arm_plan_module._combine_polarizations_device(
            amplitude_plus,
            amplitude_cross,
            w_plus_device,
            w_cross_device,
        )
        device_plan = baseline_plan.shift_plan._device_plan_arrays()

        baseline_core = (
            time_shift_jax_module
            ._assemble_shift_target_batch_chunked_lagblock_analytic_parity_core_c64
        )
        baseline_compiled = baseline_core.lower(
            unshifted_device,
            device_plan["delays"],
            device_plan["ell_all"],
            baseline_plan.shift_plan.offset,
            device_plan["Tl_all"],
            device_plan["Tp_all"],
            jnp.asarray(float(wdm.dF), dtype=jnp.float32),
            BASELINE_ROW_CHUNK,
            BASELINE_LAG_BLOCK,
        ).compile()
        memory_rows.append({
            "route": "production_batch",
            "link_block_size": 1,
            "row_chunk_size": BASELINE_ROW_CHUNK,
            **memory_analysis_to_dict(baseline_compiled.memory_analysis()),
        })

        candidate_core = (
            time_shift_jax_module
            ._assemble_shift_target_batch_chunked_lagblock_linkblock_analytic_parity_core_c64
        )
        for link_block_size, row_chunk_size in SAFE_LINKBLOCK_CANDIDATES:
            candidate_compiled = candidate_core.lower(
                unshifted_device,
                device_plan["delays"],
                device_plan["ell_all"],
                baseline_plan.shift_plan.offset,
                device_plan["Tl_all"],
                device_plan["Tp_all"],
                jnp.asarray(float(wdm.dF), dtype=jnp.float32),
                int(row_chunk_size),
                BASELINE_LAG_BLOCK,
                int(link_block_size),
            ).compile()
            memory_rows.append({
                "route": "analytic_linkblock",
                "link_block_size": int(link_block_size),
                "row_chunk_size": int(row_chunk_size),
                **memory_analysis_to_dict(candidate_compiled.memory_analysis()),
            })
    except Exception as exc:
        print("Compiled memory analysis unavailable:", repr(exc))

memory_table = show_rows(memory_rows)


## 9. Combined waveform-stage benchmark

This compares the current sequential-DWT/production-shift route with the
parallel-DWT route combined with the fastest numerically valid link-block
candidate.  It measures the complete waveform-dependent stage from padded time
series to six device-resident one-arm links.


In [ ]:
# --- 9. Combined DWT + one-arm timing ---

valid_linkblocks = [
    row for row in linkblock_rows
    if np.isfinite(row["rel_l2"]) and row["rel_l2"] <= MAX_REL_L2
]

pipeline_result = None
if valid_linkblocks:
    best_linkblock = min(
        valid_linkblocks,
        key=lambda row: row["candidate_median_s"],
    )
    best_link_block_size = int(best_linkblock["link_block_size"])
    best_row_chunk_size = int(best_linkblock["row_chunk_size"])

    def run_baseline_pipeline():
        context = _build_wdm_dT_context_from_wdm(
            times_pad,
            hplus_td,
            hcross_td,
            wdm,
            parallel_wdm_transforms=False,
        )
        return baseline_plan.apply_batch_device_materialized(
            context["w_t_plus"],
            context["w_t_cross"],
        )

    def run_candidate_pipeline():
        context = _build_wdm_dT_context_from_wdm(
            times_pad,
            hplus_td,
            hcross_td,
            wdm,
            parallel_wdm_transforms=True,
        )
        return baseline_plan.apply_batch_device_materialized_linkblock(
            context["w_t_plus"],
            context["w_t_cross"],
            link_block_size=best_link_block_size,
            row_chunk_size=best_row_chunk_size,
            lag_block_size=BASELINE_LAG_BLOCK,
        )

    for _ in range(PIPELINE_WARMUPS):
        block_result(run_baseline_pipeline())
        block_result(run_candidate_pipeline())

    candidate_output = run_candidate_pipeline()
    block_result(candidate_output)
    pipeline_comparison = compare_device_batches(
        candidate_output,
        run_baseline_pipeline(),
    )

    baseline_pipeline_times = []
    candidate_pipeline_times = []
    for repeat_index in range(PIPELINE_REPEATS):
        order = (
            (("baseline", run_baseline_pipeline), ("candidate", run_candidate_pipeline))
            if repeat_index % 2 == 0
            else (("candidate", run_candidate_pipeline), ("baseline", run_baseline_pipeline))
        )
        for label, function in order:
            started = time.perf_counter()
            output = function()
            block_result(output)
            elapsed = time.perf_counter() - started
            if label == "baseline":
                baseline_pipeline_times.append(float(elapsed))
            else:
                candidate_pipeline_times.append(float(elapsed))

    pipeline_assessment = assess_paired_candidate(
        baseline_pipeline_times,
        candidate_pipeline_times,
        rel_l2=pipeline_comparison["rel_l2"],
    )
    pipeline_result = {
        "best_link_block_size": best_link_block_size,
        "best_row_chunk_size": best_row_chunk_size,
        "baseline": pipeline_assessment["baseline"],
        "candidate": pipeline_assessment["candidate"],
        "runtime_change_percent": pipeline_assessment["runtime_change_percent"],
        "speedup": (
            pipeline_assessment["baseline"]["median_s"]
            / pipeline_assessment["candidate"]["median_s"]
        ),
        "iqr_separated": pipeline_assessment["iqr_separated"],
        "max_abs": pipeline_comparison["max_abs"],
        "rel_l2": pipeline_comparison["rel_l2"],
        "decision": pipeline_assessment["decision"],
    }

    print("Best link-block:", best_link_block_size, best_row_chunk_size)
    print("Baseline pipeline median_s:", f"{pipeline_result['baseline']['median_s']:.3f}")
    print("Candidate pipeline median_s:", f"{pipeline_result['candidate']['median_s']:.3f}")
    print("Runtime change:", f"{pipeline_result['runtime_change_percent']:+.2f}%")
    print("rel_l2:", f"{pipeline_result['rel_l2']:.3e}")
    print("Decision:", pipeline_result["decision"])
else:
    print("No numerically valid link-block candidate was available.")


## 10. Interpretation

- A fast `link_block=1` result validates the new implementation without adding
  link concurrency.
- Improvement for blocks 2, 3, or 6 at fixed `link_block * row_chunk` supports
  extra link-level parallelism.
- Similar runtime with larger compiled temporary memory suggests the extra
  vector axis is memory-limited.
- Slower safe candidates mean the production job-serial kernel is preferable;
  the next dense experiment should then be frequency tiling rather than larger
  link blocks.
- Do not promote the experimental kernel unless relative L2 is at most
  `1e-6`, median improvement is at least 3%, and candidate q75 is below
  baseline q25.
